In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2"
import tqdm
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
from scipy.special import softmax

# device = torch.device("cuda")

In [2]:
import math
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForSequenceClassification
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)
    

class Estimator():
    def __init__(self, cuda = False, batch_size=256):
        
        TOKENIZER = "s-nlp/roberta_toxicity_classifier"
        MODEL = "s-nlp/roberta_toxicity_classifier"
        BATCH_SIZE = batch_size
        self.chunk_size = 0.2
        
        self.tokenizer = AutoTokenizer.from_pretrained(TOKENIZER, use_fast=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
        self.cuda = cuda
        if cuda:
            self.model.cuda()
        
        self.training_args = TrainingArguments(  
            output_dir='./results',                   # output directory
            num_train_epochs=1,                  # total number of training epochs
            per_device_eval_batch_size=BATCH_SIZE,    # batch size for evaluation
        )

    def data_iterator(self, train_x, chunk_size = 500000):
        if chunk_size < 500000:
            chunk_size = 500000
        n_batches = math.ceil(len(train_x) / chunk_size)
        for idx in range(n_batches):
            x = train_x[idx *chunk_size:(idx+1) * chunk_size]
            yield x

    #eval_data is a list of input or a pandas frame
    def prepare_dataset(self, eval_data, max_length = 100):
        if type(eval_data) == list:
            print('start tokenizing %d lines of text'%len(eval_data))
            eval_encodings = self.tokenizer(eval_data, truncation=True, max_length = max_length, padding=True)
            eval_dataset = MyDataset(eval_encodings, [0]*len(eval_data))
            return eval_dataset
        
        
    def predict(self, eval_data, max_length = 100):      
        
        eval_iterator = self.data_iterator(eval_data, chunk_size=int(len(eval_data)*self.chunk_size))
        eval_preds = []
        
        for x in tqdm(eval_iterator):   
            trainer = Trainer(
                model=self.model,                         # the instantiated 🤗 Transformers model to be trained
                args=self.training_args,                       # training arguments, defined above
            )
            eval_dataset = self.prepare_dataset(x, max_length)
            eval_preds_raw, eval_labels , _ = trainer.predict(eval_dataset)
            eval_preds += [it[0] for it in eval_preds_raw]
        
        return eval_preds

In [3]:
def get_toxicity(texts):
    # os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
    # os.environ["CUDA_VISIBLE_DEVICES"]="0"
    model = Estimator(cuda = True)
    return model.predict(texts, max_length=128)

In [ ]:
file_list = os.listdir("/shared/4/projects/research-jam-2024/working-dir/monthly-posts-cleaned/")
for i in range(len(file_list) - 1, -1, -1):
  if "2009" in file_list[i] and ("-06" in file_list[i] or "-08" in file_list[i] or "-09" in file_list[i] or "-12" in file_list[i]):
      print(file_list[i])
      file = os.path.join("/shared/4/projects/research-jam-2024/working-dir/monthly-posts-cleaned/", file_list[i])
      df = pd.read_csv(file, sep='\t')
      df = pd.DataFrame({"id":df.message_id.copy(), "content":df.message_body_clean.copy()})
      df = df.dropna()
      content = df.content.to_list()
      ids = df.id.to_list()
      content = df['content'].to_list()
      pred = np.round(softmax(get_toxicity(content)), 3)
      
      
      df_toxicity = pd.DataFrame({"id":ids, "content":content, "pred":pred})
      new_filename = "/shared/4/projects/research-jam-2024/working-dir/toxicity/" + file_list[i] 
      df_toxicity.to_csv(new_filename, sep='\t', index=False)
      del df_toxicity, df, content, ids, pred

en.2009-09.tsv


<ipython-input-4-f7b3cd233f3f>:6: DtypeWarning: Columns (0,1,2,3,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep='\t')
/opt/anaconda/lib/python3.9/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a 

start tokenizing 500000 lines of text


/opt/anaconda/lib/python3.9/site-packages/torch/nn/parallel/data_parallel.py:33: UserWarning: 
    There is an imbalance between your GPUs. You may want to exclude GPU 2 which
    has less than 75% of the memory or cores of GPU 0. You can do so by setting
    the device_ids argument to DataParallel, or by setting the CUDA_VISIBLE_DEVICES
    environment variable.
  warnings.warn(imbalance_warn.format(device_ids[min_pos], device_ids[max_pos]))
/opt/anaconda/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


1it [21:12, 1272.17s/it]/opt/anaconda/lib/python3.9/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


start tokenizing 500000 lines of text


2it [40:09, 1192.82s/it]

start tokenizing 487427 lines of text
